[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Parameters &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/stations.db` as the notebook had it when you reached the tasks: the
stations, their year of readings, and the five notes the worked examples added. Run it first. The
tasks do not depend on one another, and the last cell removes the scratch folder.


In [1]:
import math
import shutil
import sqlite3
from contextlib import closing
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}
NOTES = [
    ("Svalbard", "2025-03-02", "The sensor's heater failed, so the station sent nothing all day"),
    ("Oslo", "2025-07-14", "Mast repainted; readings from 09:00 to 11:00 may run warm"),
    ("Tromso", "2025-10-30", 'A visitor asked "is it -- really -- always this cold?"'),
    ("Bergen", "2025-11-03", "Rain gauge cleared of leaves"),
    ("Kirkenes", "2025-12-01", "Heater checked before winter; it's working"),
]


def year_of_readings():
    """Every hour of 2025 at the four stations, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
    CREATE TABLE notes (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                        day TEXT NOT NULL, note TEXT NOT NULL);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                  ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.executemany("INSERT INTO notes (station_id, day, note) VALUES (?, ?, ?)",
                  [(ids[station], day, note) for station, day, note in NOTES])
build.commit()
build.close()

print("built", DATABASE)


built scratch/stations.db


**1.** A latitude, found by a name in a variable.


In [2]:
station = "Svalbard"

with closing(sqlite3.connect(DATABASE)) as conn:
    (latitude,) = conn.execute("SELECT latitude FROM stations WHERE name = ?", (station,)).fetchone()

print(station, latitude)


Svalbard 78.22


`(station,)` is a tuple of one, and the comma is what makes it one. `[station]` would do as well.
`fetchone` returns a row, a tuple of one value here, and `(latitude,) =` unpacks it.


**2.** A note full of characters that mean something in SQL.


In [3]:
note = """Gauge's lid replaced; "no leaks" -- so far"""

with closing(sqlite3.connect(DATABASE)) as conn:
    (bergen,) = conn.execute("SELECT id FROM stations WHERE name = ?", ("Bergen",)).fetchone()
    conn.execute("INSERT INTO notes (station_id, day, note) VALUES (?, ?, ?)", (bergen, "2025-08-19", note))
    conn.commit()
    for day, text in conn.execute("SELECT day, note FROM notes WHERE station_id = ? ORDER BY day", (bergen,)):
        print(day, text)


2025-08-19 Gauge's lid replaced; "no leaks" -- so far
2025-11-03 Rain gauge cleared of leaves


Triple quotes let the Python string hold both kinds of quotation mark. The apostrophe, the double
quotes, the semicolon and the `--` all went in as characters, since the note was a parameter, and
came back exactly as written.


**3.** One named statement, run for two stations.


In [4]:
hours_between = """
    SELECT COUNT(*)
    FROM readings AS r JOIN stations AS s ON s.id = r.station_id
    WHERE s.name = :station AND r.celsius BETWEEN :low AND :high
"""

with closing(sqlite3.connect(DATABASE)) as conn:
    for values in [{"station": "Tromso", "low": -2, "high": 2}, {"station": "Bergen", "low": 18, "high": 21}]:
        hours = conn.execute(hours_between, values).fetchone()[0]
        print(f"{values['station']:<7} from {values['low']} to {values['high']}: {hours} hours")


Tromso  from -2 to 2: 1541 hours
Bergen  from 18 to 21: 589 hours


The statement's text never changed, only the dictionary did, so sqlite3 prepared it once and ran it
twice. `BETWEEN` includes both ends, so a reading of exactly 2 or exactly 18 counts.


**4.** The coldest reading of every station in a list.


In [5]:
names = ["Bergen", "Oslo", "Tromso"]
marks = ", ".join("?" * len(names))

with closing(sqlite3.connect(DATABASE)) as conn:
    coldest = conn.execute(f"""
        SELECT s.name, MIN(r.celsius)
        FROM readings AS r JOIN stations AS s ON s.id = r.station_id
        WHERE s.name IN ({marks})
        GROUP BY s.id
        ORDER BY s.name
    """, names).fetchall()

print(coldest)


[('Bergen', -4.8), ('Oslo', -6.3), ('Tromso', -9.2)]


`marks` is written from the length of the list, three question marks here, and the names themselves
go in as parameters. Adding a station to `names` adds a question mark and a value, and nothing else
changes.


**5.** `MIN` or `MAX`, chosen from a dictionary.


In [6]:
EXTREMES = {"coldest": "MIN", "warmest": "MAX"}


def extreme(conn, station, which):
    """A station's coldest or warmest reading, with the SQL function looked up in EXTREMES."""
    query = f"""
        SELECT {EXTREMES[which]}(r.celsius)
        FROM readings AS r JOIN stations AS s ON s.id = r.station_id
        WHERE s.name = ?
    """
    return conn.execute(query, (station,)).fetchone()[0]


with closing(sqlite3.connect(DATABASE)) as conn:
    print("coldest:", extreme(conn, "Oslo", "coldest"))
    print("warmest:", extreme(conn, "Oslo", "warmest"))
    try:
        extreme(conn, "Oslo", "average")
    except KeyError as error:
        print("refused:", error)


coldest: -6.3
warmest: 19.3
refused: 'average'


A function name is part of the statement, like a column name, so no placeholder can take it. The
dictionary offers exactly two, and `"average"` raised `KeyError` before any SQL was written.


**6.** A count that `None` cannot break.


In [7]:
def count_equal(conn, celsius):
    """How many readings equal celsius, where None counts the hours with no reading."""
    return conn.execute("SELECT COUNT(*) FROM readings WHERE celsius IS ?", (celsius,)).fetchone()[0]


with closing(sqlite3.connect(DATABASE)) as conn:
    print("None:", count_equal(conn, None))
    print("20.8:", count_equal(conn, 20.8))


None: 24
20.8: 1


`IS ?` matches `NULL` to `NULL` and 20.8 to 20.8, so one statement serves both. Written with `= ?`,
the count for `None` would have been 0.

Last, remove the scratch folder:


In [8]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Parameters](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/06-parameters.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
